In [1]:
%load_ext autoreload
%autoreload

In [28]:
# Import required libraries and functions
import numpy as np
import paseos
import pykep as pk
from dotmap import DotMap
import toml
from paseos import ActorBuilder, SpacecraftActor, GroundstationActor

# Import funtions from licos
import sys
import os
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
from licos.get_constellation import get_constellation
from licos.init_paseos import init_paseos, init_paseos_scenario_0, init_paseos_scenario_1, init_paseos_scenario_2


In [29]:
# Import cfg file as DotMap
path = "../cfg/old/simulation_without_training_cfg.toml"
with open(path) as cfg:
    # dynamic=False inhibits automatic generation of non-existing keys
    cfg = DotMap(toml.load(cfg), _dynamic=False)

In [16]:
################################################################
#                                                              #
#                Run this cell for scenario 0                  #
#                                                              #
################################################################

# Starting date of our simulation
t0 = pk.epoch_from_string("2018-May-18 03:21:00")  

# Initialize paseos instance
cfg = paseos.load_default_cfg()  # loading cfg to modify defaults
cfg.sim.start_time = t0.mjd2000 * pk.DAY2SEC  # convert epoch to seconds
paseos_instances = []

for i in range(2):

    if i == 0:
        sat_name = "Sentinel-2A"
        # Sentinel-2A Orbit: (accessed 2024-07-05 14:35:10 CET at https://www.n2yo.com/satellite/?s=40697)
        #   (Period: 98.6 [min], Inclination: 98.6 [deg], Apogee: 797.0 [km], Perigee: 795.2 [km])
        line1 = "1 40697U 15028A   24187.21454778  .00000211  00000-0  96982-4 0  9994"
        line2 = "2 40697  98.5684 261.2278 0001234  95.4779 264.6545 14.30817758471926"

    else:
        sat_name = "Sentinel-2B"
        # Sentinel-2A Orbit: (accessed 2024-07-05 14:36:20 CET at https://www.n2yo.com/satellite/?s=42063#results)
        #   (Period: 98.6 [min], Inclination: 98.6 [deg], Apogee: 797.0 [km], Perigee: 795.2 [km])
        line1 = "1 42063U 17013A   24187.17957938  .00000220  00000-0  10062-3 0  9994"
        line2 = "2 42063  98.5690 261.1892 0001177  94.9260 265.2057 14.30820356382832"

    # Create the local actor
    local_actor = ActorBuilder.get_actor_scaffold(
        name=sat_name, 
        actor_type=SpacecraftActor, 
        epoch=t0
    )

    # Set the orbit of the actor
    ActorBuilder.set_TLE(local_actor, line1, line2)

    # Add a communication device to the actor
    ActorBuilder.add_comm_device(
        actor=local_actor, 
        device_name="Link1", 
        bandwidth_in_kbps=1000
    )

    # Set the power devices of the actor
    # Battery from https://sentinels.copernicus.eu/documents/247904/349490/S2_SP-1322_2.pdf
    # 87Ah * 28 Volt = 8.7696e9Ws
    ActorBuilder.set_power_devices(
        actor=local_actor,
        battery_level_in_Ws=277200 * 0.5,
        max_battery_level_in_Ws=277200,
        charging_rate_in_W=20,
    )

    # Set the thermal model of the actor
    # TODO update and sanity check
    ActorBuilder.set_thermal_model(
        actor=local_actor,
        actor_mass=6.0,
        actor_initial_temperature_in_K=283.15,
        actor_sun_absorptance=0.9,
        actor_infrared_absorptance=0.5,
        actor_sun_facing_area=0.012,
        actor_central_body_facing_area=0.01,
        actor_emissive_area=0.1,
        actor_thermal_capacity=6000,
    )

    instance = paseos.init_sim(local_actor=local_actor, cfg=cfg)
    paseos_instances.append(instance)

In [25]:
################################################################
#                                                              #
#                Run this cell for scenario 1                  #
#                                                              #
################################################################

# Starting date of our simulation
t0 = pk.epoch_from_string("2018-May-18 03:21:00")  

# Initialize paseos instance
cfg = paseos.load_default_cfg()  # loading cfg to modify defaults
cfg.sim.start_time = t0.mjd2000 * pk.DAY2SEC  # convert epoch to seconds
paseos_instances = []

# Define our central body
earth = pk.planet.jpl_lp("earth")  # define our central body

# Compute the orbit of each rank
altitude = 786 * 1000  # altitude above the Earth's ground [m]
inclination = 98.62    # inclination of the orbit
nPlanes = 1            # the number of orbital planes
nSats = 8        # the number of satellites per orbital plane
planet_list, sats_pos_and_v, _ = get_constellation(
    altitude, inclination, nSats, nPlanes, t0, verbose=False
)

# Create Satellite actors and add to paseos simulation
for sat in range(nSats):
    pos, v = sats_pos_and_v[sat]  # get our position and velocity

    # Create the local actor, name will be the rank
    local_actor = ActorBuilder.get_actor_scaffold(
        name="Sat_" + str(sat), actor_type=SpacecraftActor, epoch=t0
    )
    ActorBuilder.set_orbit(
        actor=local_actor, position=pos, velocity=v, epoch=t0, central_body=earth
    )

    # Add devices to the local actor
    ActorBuilder.add_comm_device(
        actor=local_actor, device_name="Link1", bandwidth_in_kbps=1000)

    ActorBuilder.set_power_devices(
            actor=local_actor,
            battery_level_in_Ws=277200 * 0.5,
            max_battery_level_in_Ws=277200,
            charging_rate_in_W=20)

    ActorBuilder.set_thermal_model(
            actor=local_actor,
            actor_mass=6.0,
            actor_initial_temperature_in_K=283.15,
            actor_sun_absorptance=0.9,
            actor_infrared_absorptance=0.5,
            actor_sun_facing_area=0.012,
            actor_central_body_facing_area=0.01,
            actor_emissive_area=0.1,
            actor_thermal_capacity=6000)
    
    instance = paseos.init_sim(local_actor=local_actor, cfg=cfg)
    paseos_instances.append(instance)

In [30]:
################################################################
#                                                              #
#                Run this cell for scenario 2                  #
#                                                              #
################################################################

# Starting date of our simulation
t0 = pk.epoch_from_string("2018-May-18 03:21:00")  

# Initialize paseos instance
cfg = paseos.load_default_cfg()  # loading cfg to modify defaults
cfg.sim.start_time = t0.mjd2000 * pk.DAY2SEC  # convert epoch to seconds
paseos_instances = []

# Define our central body
earth = pk.planet.jpl_lp("earth")  # define our central body

# Compute the orbit of each rank (https://www.n2yo.com/satellite/?s=42987 (accessed: 2024-07-05 10:31:40))
altitude = 452 * 1000  # altitude above the Earth's ground [m]
inclination = 97.40    # inclination of the orbit
nPlanes = 1            # the number of orbital planes
nSats = 8        # the number of satellites per orbital plane
planet_list, sats_pos_and_v, _ = get_constellation(
    altitude, inclination, nSats, nPlanes, t0, verbose=False
)

# Create Satellite actors and add to paseos simulation
for sat in range(nSats):
    pos, v = sats_pos_and_v[sat]  # get our position and velocity

    # Create the local actor, name will be the rank
    local_actor = ActorBuilder.get_actor_scaffold(
        name="Sat_" + str(sat), actor_type=SpacecraftActor, epoch=t0
    )
    ActorBuilder.set_orbit(
        actor=local_actor, position=pos, velocity=v, epoch=t0, central_body=earth
    )

    # Add devices to the local actor
    ActorBuilder.add_comm_device(
        actor=local_actor, device_name="Link1", bandwidth_in_kbps=1000)

    ActorBuilder.set_power_devices(
            actor=local_actor,
            battery_level_in_Ws=277200 * 0.5,
            max_battery_level_in_Ws=277200,
            charging_rate_in_W=20)

    ActorBuilder.set_thermal_model(
            actor=local_actor,
            actor_mass=6.0,
            actor_initial_temperature_in_K=283.15,
            actor_sun_absorptance=0.9,
            actor_infrared_absorptance=0.5,
            actor_sun_facing_area=0.012,
            actor_central_body_facing_area=0.01,
            actor_emissive_area=0.1,
            actor_thermal_capacity=6000)
    
    instance = paseos.init_sim(local_actor=local_actor, cfg=cfg)
    paseos_instances.append(instance)

In [31]:
# Add Ground stations
comms_instances = []
stations = [
    ["Maspalomas", 27.7629, -15.6338, 205.1],
    ["Matera", 40.6486, 16.7046, 536.9],
    ["Svalbard", 78.9067, 11.8883, 474.0]]

for station in stations:
    gs_actor = ActorBuilder.get_actor_scaffold(
        name=station[0], actor_type=GroundstationActor, epoch=t0
    )
    ActorBuilder.set_ground_station_location(
        gs_actor,
        latitude=station[1],
        longitude=station[2],
        elevation=station[3],
        minimum_altitude_angle=5,
    )
    instance = paseos.init_sim(local_actor=gs_actor)
    comms_instances.append(instance)

# Add disaster site
ds_actor = ActorBuilder.get_actor_scaffold(
        name="Disaster Site", actor_type=GroundstationActor, epoch=t0
    )
ActorBuilder.set_ground_station_location(
    ds_actor,
    latitude=66.30893,
    longitude=23.67734,
    elevation=127,
    minimum_altitude_angle=78.08,
)
disaster_site = paseos.init_sim(local_actor=ds_actor)

In [ ]:
time_to_advance = 15  # Advance time by 1 minute at a time
min_pass_interval = 1000  # Minimum interval in seconds to consider a new pass

first_pass = None
second_pass = None
first_comm = None
comm_passes = []
last_comm_times = {}  # To track last communication times for each satellite

# Function to check LOS
def is_in_line_of_sight(instance, target):
    return instance.local_actor.is_in_line_of_sight(target.local_actor, instance.local_time)

# Function to advance time for all instances
def advance_all_instances(instances, time_to_advance):
    for instance in instances:
        instance.advance_time(time_to_advance, 0)

print(f"{instance.local_time}  (Activation)")

# Simulate the constellation for ~24 hours from the given initial epoch
for val in np.arange(0, 86400, time_to_advance):
    
    for sat_id, instance in enumerate(paseos_instances):
        
        # Check for LOS with the disaster site
        if first_pass is None and is_in_line_of_sight(instance, disaster_site):
            first_pass = (instance.local_time, sat_id)
            print(f"{instance.local_time}  ({instance.local_actor.name} - disaster site)")
            break  # Break to start tracking comms and second pass
        
    if first_pass is not None:
        for sat_id, instance in enumerate(paseos_instances):
            
            # Check for LOS with the disaster site for the second pass
            if second_pass is None and is_in_line_of_sight(instance, disaster_site):
                if (sat_id != first_pass[1]) or \
                   (instance.local_time.mjd2000 * pk.DAY2SEC - first_pass[0].mjd2000 * pk.DAY2SEC > min_pass_interval):  # Ensure some time has passed if it's the same satellite
                    
                    if first_comm is None:
                        print(f"{instance.local_time}  ({instance.local_actor.name} - disaster site - no prior comm)")
                        first_pass = (instance.local_time, sat_id)
                        
                    else:
                        #second_pass = (instance.local_time, sat_id)
                        print(f"{instance.local_time}  ({instance.local_actor.name} - disaster site)")
                        #break  # No need to check further once second pass is found
            
            # Track LOS with comm sites between the first and second pass
            if second_pass is None:
                for site_id, comm_site in enumerate(comms_instances):
                    if is_in_line_of_sight(instance, comm_site):
                        delta_t_first_pass = instance.local_time.mjd2000 * pk.DAY2SEC - first_pass[0].mjd2000 * pk.DAY2SEC
                        if delta_t_first_pass > 0:  # Ensure comm occurs after the first pass
                            if (sat_id not in last_comm_times) or \
                               (instance.local_time.mjd2000 * pk.DAY2SEC - last_comm_times[sat_id] > min_pass_interval):
                                comm_passes.append((instance.local_time, sat_id, comm_site.local_actor.name))
                                last_comm_times[sat_id] = instance.local_time.mjd2000 * pk.DAY2SEC
                                first_comm = True
                                print(f"{instance.local_time}  ({instance.local_actor.name} - {comm_site.local_actor.name})")

    # Stop the loop if we have found the first and second pass
    if first_pass and second_pass:
        break

    # Advance time for all instances
    advance_all_instances(paseos_instances, time_to_advance)


# Plot the constellation at the time of the second pass with LOS of the disaster site
paseos_instances[0].empty_known_actors()
for instance in paseos_instances[1:]:
    paseos_instances[0].add_known_actor(instance.local_actor)
for instance in comms_instances:
    paseos_instances[0].add_known_actor(instance.local_actor)
paseos_instances[0].add_known_actor(disaster_site.local_actor)
plotter = paseos.plot(paseos_instances[0], paseos.PlotType.SpacePlot)
